# 📘 Study Session 4 — Multiple Regression & Model Evaluation

---

### 🎯 Goals

By the end of this session, you will be able to:

- Understand **multiple linear regression** — modeling a target with two or more predictors.  
- Use **scikit-learn** to fit and interpret a `LinearRegression()` model.  
- Explain what **coefficients**, **intercept**, and **R²** mean in a multi-variable context.  
- Evaluate models using **Adjusted R²** and **residual plots**.  
- Identify overfitting by comparing model complexity vs. explanatory power.  
- Save and log visualizations automatically with `save_and_log_plot()` for reproducibility.

---

### 🧩 Key Concept

> Multiple regression expands the simple “line of best fit” into a *multi-dimensional plane*,  
> letting you control for several variables (e.g., bill size, gender, and party size)  
> to see how each uniquely affects the tip amount.

---

**Next:** Begin by importing your cleaned dataset (`data/processed/tips_cleaned.csv`)  
and verifying column names before encoding categorical variables.


## Setup and Data Preparation

In [ ]:
# --- 1. Import libraries ---
import os
import sys
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Manually define the correct project root path based on your previous output
CORRECT_PROJECT_ROOT = "/home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one"

# Set CWD
try:
    os.chdir(CORRECT_PROJECT_ROOT)
    print(f"✅ CWD successfully set to: {os.getcwd()}")

    # Add to sys.path for module imports (src.helpers)
    if CORRECT_PROJECT_ROOT not in sys.path:
        sys.path.append(CORRECT_PROJECT_ROOT)
        print("✅ Added project root to sys.path.")

except FileNotFoundError:
    print("❌ CRITICAL ERROR: The manually defined project path does not exist.")
    sys.exit(1)

# --- 2. Load dataset saved previously above from seaborn's GitHub mirror ---
# NOTE: Path adjusted to be relative to the Project Root CWD
df = duckdb.query("""
    SELECT * FROM read_csv_auto('data/processed/tips_cleaned.csv')
""").df()

df.info

In [ ]:
# df.info

## Gen Study Session 4 Checklist PDF

In [ ]:
# print (CORRECT_PROJECT_ROOT)
outdir = CORRECT_PROJECT_ROOT + "/../PDF"
print (outdir)

In [ ]:
# === Study Session 4 PDF generator ===
# Creates: ds-zero-to-one/reports/Study_Session_4_Checklist.pdf

# 1) Ensure reportlab is available
try:
    from reportlab.lib.pagesizes import LETTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])
    from reportlab.lib.pagesizes import LETTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch

from pathlib import Path

# 2) Output path (adjust if your repo root differs)
out_dir = Path(outdir)  # <-- change if needed
# out_dir = repo_root / "reports"
# out_dir.mkdir(parents=True, exist_ok=True)
pdf_path = out_dir / "Study_Session_4_Checklist.pdf"

# 3) Styles
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=1))
styles.add(ParagraphStyle(name="Subhead", parent=styles["Heading2"], spaceBefore=10, spaceAfter=6))
styles.add(ParagraphStyle(name="BodySm", parent=styles["BodyText"], fontSize=11, leading=14))

# 4) Content
goals = [
    "Build and interpret a multiple linear regression model.",
    "Encode categorical predictors (e.g., gender, is_smoker) correctly.",
    "Evaluate fit with R² and Adjusted R²; read residual plots.",
    "Detect issues (non-linearity, heteroscedasticity, multicollinearity).",
    "Save charts with save_and_log_plot() for reproducibility and indexing.",
]

steps = [
    "Load cleaned dataset (data/processed/tips_cleaned.csv) and inspect columns.",
    "One-hot encode categoricals: pd.get_dummies(df, columns=['gender','is_smoker'], drop_first=True).",
    "Define X = ['bill_total_usd','party_size','gender_Male','is_smoker_Yes'], y = tip_usd.",
    "Fit LinearRegression(); print coefficients, intercept, and R².",
    "Compute Adjusted R²: 1 - (1 - R2)*(n - 1)/(n - p - 1)  (n=rows, p=#features).",
    "Plot Actual vs Predicted with red dashed y=x line; save via save_and_log_plot().",
    "Plot residuals (y - y_pred) vs predicted; add horizontal line at 0.",
    "Optional: check multicollinearity (VIF) and drop/transform if needed.",
    "Write a short Markdown interpretation: which variables matter and why?",
]

vif_hint = [
    "VIF quick check (optional): for each numeric column in X:",
    "VIF ≈ 1/(1 - R²_from_regressing_that_feature_on_all_others).",
    "High VIF (> 5–10) suggests multicollinearity; consider dropping or combining.",
]

reflection = [
    "Did adding predictors improve Adjusted R² meaningfully?",
    "Which coefficients were strongest, and do they match intuition?",
    "Do residuals look randomly scattered (good) or patterned (bad)?",
    "Any signs of heteroscedasticity (funnel shape) or outliers?",
]

eli5_text = (
    "Multiple regression is like guessing tips using several clues at once: bill size, group size, "
    "and whether someone smokes. Each coefficient says how much tips change when that one clue changes, "
    "while the others stay the same. The red dashed line (y = x) shows perfect guesses—your points getting "
    "close to it means your model is doing well."
)

# 5) Build the PDF
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=LETTER,
    leftMargin=0.75*inch, rightMargin=0.75*inch,
    topMargin=0.75*inch, bottomMargin=0.75*inch,
    title="Study Session 4 — Multiple Regression & Model Evaluation",
)

story = []
story.append(Paragraph("📘 Study Session 4 — Multiple Regression & Model Evaluation", styles["TitleCenter"]))
story.append(Spacer(1, 0.2*inch))

story.append(Paragraph("🎯 Goals", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(g, styles["BodySm"])) for g in goals], bulletType="bullet"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("🧠 Steps", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(s, styles["BodySm"])) for s in steps], bulletType="1"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("🧪 Multicollinearity (VIF) — Optional", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(v, styles["BodySm"])) for v in vif_hint], bulletType="bullet"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("💭 Reflection Questions", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(q, styles["BodySm"])) for q in reflection], bulletType="bullet"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("✨ ELI5 Summary", styles["Subhead"]))
story.append(Paragraph(eli5_text, styles["BodySm"]))

doc.build(story)
print(f"✅ PDF written to: {pdf_path}")


In [ ]:
df_enc = pd.get_dummies(df, columns=["gender", "is_smoker"], drop_first=True)

In [ ]:
# Check if the column exists
'is_smoker' in df.columns

In [ ]:
# View unique values
df['is_smoker'].unique()

In [ ]:
# Count how many of each
df['is_smoker'].value_counts()

In [ ]:

# See first few rows of that column
df[['is_smoker']].head()


In [ ]:
import pandas as pd

# 1) Work on a copy
df2 = df.copy()

# 2) Normalize categorical text (handles typos/case like 'M', 'male ', 'YES', True/False, etc.)
def norm_gender(x):
    x = str(x).strip().lower()
    if x in {"m", "male"}: return "Male"
    if x in {"f", "female"}: return "Female"
    return pd.NA

def norm_smoker(x):
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}: return "Yes"
    if s in {"no", "n", "false", "0"}: return "No"
    return pd.NA

df2["gender"] = df2["gender"].apply(norm_gender)
df2["is_smoker"] = df2["is_smoker"].apply(norm_smoker)

# 3) Lock categories so dummies are predictable even if a level is absent in this sample
df2["gender"] = pd.Categorical(df2["gender"], categories=["Female", "Male"])
df2["is_smoker"] = pd.Categorical(df2["is_smoker"], categories=["No", "Yes"])

# 4) Make dummies (drop_first=True => reference levels Female, No)
df_enc = pd.get_dummies(df2, columns=["gender", "is_smoker"], drop_first=True)

# 5) Guarantee expected columns exist (in case this subset lacks one level)
for col in ["gender_Male", "is_smoker_Yes"]:
    if col not in df_enc.columns:
        df_enc[col] = 0

# 6) Sanity checks
print("Dummy columns present:", [c for c in df_enc.columns if c.startswith(("gender_", "is_smoker_"))])
print("gender unique:", df2["gender"].unique())
print("is_smoker unique:", df2["is_smoker"].unique())


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df_enc[["bill_total_usd", "party_size", "gender_Male", "is_smoker_Yes"]]
y = df_enc["tip_usd"]

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

print("Intercept:", round(model.intercept_, 3))
for name, coef in zip(X.columns, model.coef_):
    print(f"{name}: {coef:.3f}")
print("R²:", round(r2_score(y, y_pred), 3))


# Gemini ELI5 the Output: Predicting Tip Amounts 💰

That output is a **recipe** for predicting the dollar amount of a **tip** based on several factors about the customer and their bill. It comes from a math tool called a **Linear Regression Model**.

***

## The Prediction Recipe

Imagine you're trying to guess the tip amount at a restaurant. This model tells you how much to start with (the Intercept) and how to adjust that guess for each factor (the coefficients).

The model gives you a simple math equation:

$$\text{Tip Amount} = 0.722 + (0.094 \times \text{Bill Total}) + (0.180 \times \text{Party Size}) + (\text{Adjustments for Gender and Smoking})$$

| Term | Value | ELI5 Meaning |
| :--- | :--- | :--- |
| **Intercept** | **0.722** | **The Starting Guess.** If the bill was $\$0$, the party size was 0, and we ignored gender/smoking, the model predicts a base tip of **72 cents**. |
| **bill\_total\_usd** | **+0.094** | **The Bill's Impact.** For every **one dollar** the bill goes up, the predicted tip goes up by about **9.4 cents**. This is the biggest factor. |
| **party\_size** | **+0.180** | **The Group's Impact.** For every **extra person** in the party, the predicted tip goes up by **18 cents**. |
| **gender\_Male** | **-0.027** | **The Gender Adjustment.** If the customer is a **Male**, the predicted tip is **2.7 cents lower** (relative to Female). |
| **is\_smoker\_Yes** | **-0.084** | **The Smoking Adjustment.** If the customer is a **Smoker**, the predicted tip is **8.4 cents lower** (relative to a non-smoker). |

***

## Model Confidence Score

| Term | Value | ELI5 Meaning |
| :--- | :--- | :--- |
| **R²** | **0.469** | **The Confidence Score.** This score of **46.9%** means the model can explain **46.9%** of the differences in actual tip amounts just by using these four factors. The rest is due to things the model doesn't know. |

***

## Example Calculation

Let's predict the tip for a **Male smoker** with a **$\$30$ bill** in a party of **2**:

$$\text{Tip} = 0.722 + (0.094 \times 30) + (0.180 \times 2) + (-0.027 \times 1) + (-0.084 \times 1)$$

$$\text{Tip} = 0.722 + 2.82 + 0.36 - 0.027 - 0.084$$

$$\text{Tip} \approx \$3.79$$

The model predicts a tip of about **$\$3.79$**.

### 📊 chatgpt Regression Results — ELI5 Interpretation

| Term | Meaning | Plain-English Explanation |
|------|----------|----------------------------|
| **Intercept = 0.722** | The baseline tip when all predictors are zero. | If someone ordered a \$0 meal (theoretical), the model still “expects” about a **\$0.72 tip** baseline. |
| **bill_total_usd = 0.094** | Tip change per dollar of bill. | For every **\$1 increase** in the bill, the tip rises by roughly **\$0.09**. So a \$10 bigger bill adds about **\$0.94** to the tip. |
| **party_size = 0.180** | Tip change per extra person. | Each additional person in the group increases the tip by about **\$0.18**, on average. |
| **gender_Male = -0.027** | Adjustment for male servers (vs. female). | Male servers earn around **\$0.03 less** per tip, holding other factors constant. |
| **is_smoker_Yes = -0.084** | Adjustment for smoking tables. | Smoking tables tip roughly **\$0.08 less** on average than non-smoking tables. |
| **R² = 0.469** | Goodness of fit. | The model explains about **47% of the variation in tips** — a decent but not perfect predictor. |

---

### 🧠 ELI5 Summary

> Imagine you’re guessing someone’s tip using clues:  
> how big the bill was, how many people were there, whether they smoke, and whether the server was male.  
> Each clue nudges your guess a little up or down.  
>  
> - Bigger bills → higher tips.  
> - Bigger parties → higher tips.  
> - Smokers and male servers → slightly lower tips.  
>  
> Your guessing game is right about **half the time (47%)**,  
> so it’s helpful but not perfect.


## save markdown text to PDF

In [ ]:
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from pathlib import Path

# === CONFIG ===

# repo_root = Path("/home/rtackett/ds-zero-to-one")
# out_dir = repo_root / "reports"
# out_dir.mkdir(parents=True, exist_ok=True)
# pdf_path = out_dir / "Regression_ELI5_Explanation.pdf"

out_dir = Path(outdir)  # <-- change if needed
pdf_path = out_dir / "Regression_ELI5_Explanation.pdf"

# === STYLES ===
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Body", parent=styles["BodyText"], fontSize=11, leading=14))

# === TEXT (paste your Markdown cell here but without table formatting) ===
content = """
📊 **Regression Results — ELI5 Interpretation**

Intercept = 0.722  
bill_total_usd = 0.094  
party_size = 0.180  
gender_Male = -0.027  
is_smoker_Yes = -0.084  
R² = 0.469  

💡 *Plain-English summary:*  
For every $1 higher bill, the tip goes up by about $0.09.  
Larger parties tip more, while smoking tables and male servers tip slightly less.  
The model explains about 47% of tip variation — decent, not perfect.

🧠 **ELI5 Summary:**  
Imagine you’re guessing someone’s tip using clues: bill size, group size, smoking, and server gender.  
Each clue nudges your guess a little up or down. Bigger bills and groups mean higher tips.  
Smokers and male servers slightly lower. Your guessing game is right about half the time.
"""

# === BUILD PDF ===
doc = SimpleDocTemplate(str(pdf_path), pagesize=LETTER, topMargin=0.75*inch, bottomMargin=0.75*inch)
story = [Paragraph(content.replace("\n", "<br/>"), styles["Body"])]
doc.build(story)

print(f"✅ Saved to {pdf_path}")


In [ ]:
# === Regression ELI5 Markdown → PDF ===
# Output: ds-zero-to-one/reports/Regression_ELI5_Explanation.pdf

from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from pathlib import Path


out_dir = Path(outdir)  # <-- change if needed
pdf_path = out_dir / "Regression_ELI5_Explanation2.pdf"

# --- 2) Styles ---
styles = getSampleStyleSheet()

# styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=1))
# styles.add(ParagraphStyle(name="Subhead", parent=styles["Heading2"], spaceBefore=10, spaceAfter=6))
# styles.add(ParagraphStyle(name="BodySm", parent=styles["BodyText"], fontSize=11, leading=14))

styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=1))
styles.add(ParagraphStyle(name="BodySm", parent=styles["BodyText"], fontSize=10.5, leading=14))
styles.add(ParagraphStyle(name="TableCell", parent=styles["BodyText"], fontSize=9.5, leading=12))

# --- 3) Table content ---
table_data = [
    ["Term", "Meaning", "Plain-English Explanation"],
    ["Intercept = 0.722", "Baseline tip when all predictors are zero.",
     "If someone ordered a $0 meal (theoretical), the model still “expects” about a $0.72 tip baseline."],
    ["bill_total_usd = 0.094", "Tip change per dollar of bill.",
     "For every $1 increase in the bill, the tip rises by roughly $0.09. So a $10 bigger bill adds about $0.94 to the tip."],
    ["party_size = 0.180", "Tip change per extra person.",
     "Each additional person in the group increases the tip by about $0.18, on average."],
    ["gender_Male = -0.027", "Adjustment for male servers (vs. female).",
     "Male servers earn around $0.03 less per tip, holding other factors constant."],
    ["is_smoker_Yes = -0.084", "Adjustment for smoking tables.",
     "Smoking tables tip roughly $0.08 less on average than non-smoking tables."],
    ["R² = 0.469", "Goodness of fit.",
     "The model explains about 47% of the variation in tips — a decent but not perfect predictor."],
]

# Convert all strings to Paragraphs (so wrapping works)
for i in range(1, len(table_data)):
    table_data[i] = [Paragraph(str(cell), styles["TableCell"]) for cell in table_data[i]]
table_data[0] = [Paragraph(f"<b>{h}</b>", styles["TableCell"]) for h in table_data[0]]

# --- 4) Table layout with proper widths ---
table = Table(table_data, colWidths=[1.6*inch, 2.0*inch, 3.9*inch])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("ALIGN", (0, 0), (-1, -1), "LEFT"),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
    ("LEFTPADDING", (0, 0), (-1, -1), 5),
    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 5),
]))

# --- 4) ELI5 summary text ---
eli5_text = """
🧠 <b>ELI5 Summary</b><br/>
Imagine you’re guessing someone’s tip using clues: bill size, group size, smoking, and server gender.<br/>
Each clue nudges your guess a little up or down.<br/><br/>
<ul>
<li>Bigger bills → higher tips.</li>
<li>Bigger parties → higher tips.</li>
<li>Smokers and male servers → slightly lower tips.</li>
</ul>
Your guessing game is right about <b>half the time (47%)</b>, so it’s helpful but not perfect.
"""

# --- 5) Build PDF ---
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=LETTER,
    leftMargin=0.75*inch,
    rightMargin=0.75*inch,
    topMargin=0.75*inch,
    bottomMargin=0.75*inch,
    title="Regression ELI5 Explanation (Wrapped)",
)

story = [
    Paragraph("📊 Regression Results — ELI5 Interpretation", styles["TitleCenter"]),
    Spacer(1, 0.25*inch),
    table,
    Spacer(1, 0.3*inch),
    Paragraph(eli5_text, styles["BodySm"]),
]

doc.build(story)
print(f"✅ PDF saved to: {pdf_path}")